# Speculative Decoding Benchmark (HuggingFace Transformers)

Альтернатива SGLang-серверному подходу. Всё работает **в процессе ноутбука** — никакого отдельного сервера.

**Как это работает:**
```python
model.generate(assistant_model=draft_model, num_assistant_tokens=K)
```

HF Transformers поддерживает speculative decoding через `assisted generation`:
1. Draft-модель авторегрессивно генерирует K токенов
2. Target-модель верифицирует все K токенов за один forward pass
3. Принятые токены сохраняются, отвергнутые — отбрасываются

## Метрики
- **acceptance_length** — среднее кол-во принятых токенов за шаг верификации
- **step_time_ms** — среднее время одного шага (draft + verify)
- **speed_tok_s** — токены в секунду

## Поддержка квантизации draft-модели
- `None` — fp16/bf16
- `"4bit"` — bitsandbytes NF4
- `"8bit"` — bitsandbytes INT8

## Требования
- Google Colab / DataSphere / Jupyter с GPU
- `bench_hf.py` рядом с ноутбуком

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

Sat Mar 21 15:03:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 2: Install dependencies
# After install, RESTART KERNEL, then skip this cell.

%pip install transformers>=4.38.0 accelerate bitsandbytes
%pip install torch  # usually pre-installed in Colab
%pip install pandas matplotlib numpy

## Configuration

Настройте:
1. **TARGET_MODEL** — основная модель (fp16/bf16)
2. **DRAFT_CONFIGS** — список драфт-конфигураций для сравнения
3. **NUM_ASSISTANT_TOKENS_LIST** — кол-во токенов, которые драфт генерирует за шаг

Каждая конфигурация в `DRAFT_CONFIGS`:
- `model_path` — путь к модели (None = baseline без спекуляции)
- `quantization` — "4bit", "8bit" или None
- `label` — название для графиков

In [ ]:
# ============================================================
# Configuration — edit this cell
# ============================================================

TARGET_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TARGET_QUANTIZATION = None

DRAFT_CONFIGS = [
    {
        "model_path": "Qwen/Qwen2.5-1.5B-Instruct",
        "quantization": "4bit",
        "label": "self-4bit",
    },
    {
        "model_path": "Qwen/Qwen2.5-0.5B-Instruct",
        "quantization": None,
        "label": "0.5B-fp16",
    },
    {
        "model_path": "Qwen/Qwen2.5-0.5B-Instruct",
        "quantization": "4bit",
        "label": "0.5B-4bit",
    },
]

# 0 = baseline (no speculation), included automatically
NUM_ASSISTANT_TOKENS_LIST = [0, 3, 5, 8]

MAX_NEW_TOKENS = 256
NUM_PROMPTS = 8

OUTPUT_FILE = "bench_results.jsonl"

print(f"Target: {TARGET_MODEL} (quant={TARGET_QUANTIZATION})")
for dc in DRAFT_CONFIGS:
    print(f"  draft: {dc['label']} — {dc['model_path']} (quant={dc.get('quantization')})")
print(f"num_assistant_tokens: {NUM_ASSISTANT_TOKENS_LIST}, max_new_tokens={MAX_NEW_TOKENS}, num_prompts={NUM_PROMPTS}")

In [ ]:
# Cell 4: Load target model and tokenizer

import torch
from bench_hf import load_model, load_tokenizer, get_gpu_memory_info, get_model_memory_mb

print("Loading target model...")
target_model = load_model(
    TARGET_MODEL,
    quantization=TARGET_QUANTIZATION
)
tokenizer = load_tokenizer(TARGET_MODEL)

print(f"Target model loaded: {get_model_memory_mb(target_model):.0f} MB params")
print(f"GPU memory: {get_gpu_memory_info()}")
print(f"Model device: {target_model.device}")
print(f"Model dtype: {target_model.dtype}")

Loading target model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Target model loaded: 2944 MB params
GPU memory: {'allocated_mb': 2944.4, 'reserved_mb': 3000.0, 'total_mb': 14912.7}
Model device: cuda:0
Model dtype: torch.bfloat16


## Run Benchmarks

Для каждой draft-конфигурации:
1. Загрузка draft-модели (если есть)
2. Для каждого `num_assistant_tokens`:
   - Warmup
   - Benchmark (N промптов)
3. Выгрузка draft-модели, освобождение GPU памяти

Target-модель загружается один раз и переиспользуется.

In [ ]:
import os
from bench_hf import run_multi_draft_benchmark

# Clear previous results
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)
    print(f"Removed old {OUTPUT_FILE}")

# Run all benchmarks
results = run_multi_draft_benchmark(
    target_model=target_model,
    tokenizer=tokenizer,
    draft_configs=DRAFT_CONFIGS,
    num_assistant_tokens_list=NUM_ASSISTANT_TOKENS_LIST,
    max_new_tokens=MAX_NEW_TOKENS,
    num_prompts=NUM_PROMPTS,
    output_file=OUTPUT_FILE,
)

print(f"\nDone! {len(results)} configurations benchmarked.")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



############################################################
  Draft config: baseline
############################################################

  [1/1] baseline
  Warming up...
  Running (8 prompts, max_new_tokens=256)...
  -> speed=22.66 tok/s, acc_length=1.004, step_time=44.30ms

############################################################
  Draft config: self-4bit
  model=Qwen/Qwen2.5-1.5B-Instruct, quant=4bit
############################################################
  Loading draft model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Draft model loaded (1070 MB parameters)
  GPU: {'allocated_mb': 4071.2, 'reserved_mb': 5798.0, 'total_mb': 14912.7}

  [1/4] baseline
  Warming up...
  Running (8 prompts, max_new_tokens=256)...


Passing `generation_config` together with generation-related arguments=({'num_assistant_tokens', 'num_assistant_tokens_schedule'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'use_cache', 'min_new_tokens', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingfa

...

  Running (8 prompts, max_new_tokens=256)...


Both `max_new_tokens` (=20) and `max_length`(=279) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=279) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  -> speed=10.08 tok/s, acc_length=3.368, step_time=334.12ms
  Draft model unloaded, GPU memory freed.

############################################################
  Draft config: 0.5B-4bit
  model=Qwen/Qwen2.5-0.5B-Instruct, quant=4bit
############################################################
  Loading draft model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  Draft model loaded (430 MB parameters)
  GPU: {'allocated_mb': 3394.3, 'reserved_mb': 3838.0, 'total_mb': 14912.7}

  [1/4] baseline
  Warming up...
  Running (8 prompts, max_new_tokens=256)...


## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print("=" * 80)
print(f"Benchmark Results — {TARGET_MODEL}")
print("=" * 80)

# Show key columns
display_cols = [
    "draft_label", "num_assistant_tokens", "is_baseline",
    "speed_tok_s", "acc_length", "step_time_ms",
    "total_output_tokens", "total_time_s",
]
display(df[display_cols])

In [ ]:
import matplotlib.pyplot as plt

df["label"] = df.apply(
    lambda r: f"{r['draft_label']}\nnat={int(r['num_assistant_tokens'])}",
    axis=1,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["#2196F3" if r["is_baseline"] else "#4CAF50" for _, r in df.iterrows()]

axes[0].bar(range(len(df)), df["speed_tok_s"], color=colors)
axes[0].set_xticks(range(len(df)))
axes[0].set_xticklabels(df["label"], fontsize=7, rotation=45, ha="right")
axes[0].set_ylabel("Speed (tokens/s)")
axes[0].set_title("Decoding Speed")
baseline_speed_line = df[df["is_baseline"]]["speed_tok_s"].mean()
axes[0].axhline(y=baseline_speed_line, color="red", linestyle="--", alpha=0.5, label="baseline")
axes[0].legend(fontsize=8)

axes[1].bar(range(len(df)), df["acc_length"], color=colors)
axes[1].set_xticks(range(len(df)))
axes[1].set_xticklabels(df["label"], fontsize=7, rotation=45, ha="right")
axes[1].set_ylabel("Acceptance Length")
axes[1].set_title("Acceptance Length (higher = better)")

axes[2].bar(range(len(df)), df["step_time_ms"], color=colors)
axes[2].set_xticks(range(len(df)))
axes[2].set_xticklabels(df["label"], fontsize=7, rotation=45, ha="right")
axes[2].set_ylabel("Step Time (ms)")
axes[2].set_title("Step Time (lower = better)")

plt.suptitle(f"Speculative Decoding Benchmark — {TARGET_MODEL}", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("bench_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: bench_results.png")

In [ ]:
baseline_rows = df[df["is_baseline"]]
spec_rows = df[~df["is_baseline"]].copy()

baseline_speed = baseline_rows["speed_tok_s"].mean()
spec_rows["speedup"] = spec_rows["speed_tok_s"] / baseline_speed

print(f"Baseline speed: {baseline_speed:.2f} tok/s")
print("-" * 70)
for _, row in spec_rows.iterrows():
    print(
        f"  {row['draft_label']:>12s}, nat={int(row['num_assistant_tokens']):>2d} "
        f"-> {row['speedup']:.2f}x speedup "
        f"({row['speed_tok_s']:.1f} vs {baseline_speed:.1f} tok/s), "
        f"acc_length={row['acc_length']:.3f}"
    )

best = spec_rows.loc[spec_rows["speedup"].idxmax()]
print(f"\nBest: {best['draft_label']}, nat={int(best['num_assistant_tokens'])} -> {best['speedup']:.2f}x speedup")

In [ ]:
pivot = spec_rows.pivot_table(
    index="draft_label",
    columns="num_assistant_tokens",
    values="speedup",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, max(3, len(pivot) * 0.8)))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=0.5)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("num_assistant_tokens")
ax.set_ylabel("Draft config")
ax.set_title(f"Speedup vs Baseline — {TARGET_MODEL}")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.2f}x", ha="center", va="center", fontsize=11)

plt.colorbar(im, label="Speedup")
plt.tight_layout()
plt.savefig("speedup_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: speedup_heatmap.png")

## Save / Load Results

Результаты сохраняются в `bench_results.jsonl` после каждой конфигурации.
Если ноутбук перезапущен, можно загрузить из файла.

In [ ]:
# Reload results from file (useful after kernel restart)
# import json
# import pandas as pd
# results = [json.loads(line) for line in open("bench_results.jsonl")]
# df = pd.DataFrame(results)
# display(df)